# Same-Modal Retrieval: optical to optical

Standalone Kaggle training and inference notebook for the fixed optical-to-optical task. Relevance means at least one shared BigEarthNet 19-class label.

## 1. Packages, variables, and configuration

In [ ]:
import importlib.util, subprocess, sys
packages={"timm":"timm>=1.0.15","rasterio":"rasterio>=1.3","huggingface_hub":"huggingface_hub>=0.27","safetensors":"safetensors>=0.4"}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])

import ast, hashlib, json, math, os, random, time, warnings
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
import timm, torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file as load_safetensors
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

@dataclass
class Config:
    DATA_ROOT: Path=Path("/kaggle/input/datasets/glitchr/bigearthnet-48k-subset")
    OUTPUT_DIR: Path=Path("/kaggle/working/outputs/optical-to-optical")
    MODEL_NAME: str="vit_base_patch8_224"
    IMAGE_SIZE: int=120
    EMBED_DIM: int=256
    OPTICAL_INIT: str="imagenet"  # imagenet, adapt_reben_s2, random
    HF_LOCAL_DIR: str|None=None
    HF_TOKEN: str|None=None
    BATCH_SIZE: int=64
    EVAL_BATCH_SIZE: int=128
    NUM_WORKERS: int=2
    EPOCHS: int=20
    LR: float=3e-4
    WEIGHT_DECAY: float=1e-4
    TEMPERATURE: float=.07
    PAIRED_WEIGHT: float=.7
    SEMANTIC_WEIGHT: float=.3
    PATIENCE: int=5
    SEED: int=42
    MAX_TRAIN_SAMPLES: int|None=None
    MAX_EVAL_SAMPLES: int|None=None
    SIM_CHUNK: int=512
    RESUME: bool=True
    RUN_TRAINING: bool=True
    QUICK_MODE: bool=False

cfg=Config()
cfg.S1_ROOT=cfg.DATA_ROOT/"BigEarthNet-S1"; cfg.S2_ROOT=cfg.DATA_ROOT/"BigEarthNet-S2"
cfg.METADATA=cfg.DATA_ROOT/"ben_subset.csv"; cfg.CKPT_DIR=cfg.OUTPUT_DIR/"checkpoints"; cfg.RESULTS_DIR=cfg.OUTPUT_DIR/"results"
if cfg.QUICK_MODE: cfg.EPOCHS=2; cfg.MAX_TRAIN_SAMPLES=512; cfg.MAX_EVAL_SAMPLES=256
for p in (cfg.OUTPUT_DIR,cfg.CKPT_DIR,cfg.RESULTS_DIR): p.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED); torch.cuda.manual_seed_all(cfg.SEED)
TASK_NAME="optical-to-optical"; QUERY_MODALITY="optical"; GALLERY_MODALITY="optical"; SAME_MODAL=True
MODALITIES={"optical":{"bands":["B04","B03","B02"],"source":"s2","repo":null}}
REBEN_REPOS={}
BEST=cfg.CKPT_DIR/"best.pth"; LAST=cfg.CKPT_DIR/"last.pth"
print(TASK_NAME,DEVICE,MODALITIES)

## 2. Utility functions and metadata validation

In [ ]:
LABELS=["Urban fabric","Industrial or commercial units","Arable land","Permanent crops","Pastures","Complex cultivation patterns","Land principally occupied by agriculture, with significant areas of natural vegetation","Agro-forestry areas","Broad-leaved forest","Coniferous forest","Mixed forest","Natural grassland and sparsely vegetated areas","Moors, heathland and sclerophyllous vegetation","Transitional woodland, shrub","Beaches, dunes, sands","Inland wetlands","Coastal wetlands","Inland waters","Marine waters"]
L2I={x:i for i,x in enumerate(LABELS)}
MEAN={"VV":-12.6438637,"VH":-19.3525581,"B02":438.3721,"B03":614.0557,"B04":588.4096,"B05":942.8433,"B06":1769.9316,"B07":2049.5515,"B08":2193.2920,"B8A":2235.5566,"B11":1568.2268,"B12":997.7325}
STD={"VV":5.1334939,"VH":5.5905056,"B02":607.0269,"B03":603.2968,"B04":684.5688,"B05":738.4327,"B06":1100.4561,"B07":1275.8054,"B08":1369.3717,"B8A":1356.5441,"B11":1070.1613,"B12":813.5276}

def parse_labels(value):
    if value is None or (isinstance(value,float) and pd.isna(value)): return []
    if isinstance(value,(list,tuple,set,np.ndarray)): items=list(value)
    else:
        text=str(value).strip(); items=None
        for parser in (json.loads,ast.literal_eval):
            try:
                parsed=parser(text); items=list(parsed) if isinstance(parsed,(list,tuple,set)) else [parsed]; break
            except Exception: pass
        if items is None:
            quoted=__import__("re").findall(r"""['"]([^'"]+)['"]""",text)
            items=quoted or [x.strip() for x in __import__("re").split(r"[;,|]",text) if x.strip()]
    out=[]
    for x in items:
        x=str(x).strip().strip("'\"")
        if x in L2I: out.append(x)
        elif x.isdigit() and 0<=int(x)<19: out.append(LABELS[int(x)])
    return sorted(set(out),key=LABELS.index)

def s1_path(name):
    return cfg.S1_ROOT/"_".join(str(name).split("_")[:-3])/str(name)
def s2_path(patch_id):
    return cfg.S2_ROOT/"_".join(str(patch_id).split("_")[:-2])/str(patch_id)
def patch_path(modality,row):
    return s1_path(row["s1_name"]) if MODALITIES[modality]["source"]=="s1" else s2_path(row["patch_id"])
def band_map(folder,bands):
    found={}
    for path in list(folder.glob("*.tif"))+list(folder.glob("*.tiff")):
        stem=path.stem.upper()
        for band in sorted(bands,key=len,reverse=True):
            if stem==band or stem.endswith("_"+band) or "_"+band+"_" in stem: found[band]=path; break
    return found
def split_for(value):
    u=int(hashlib.md5(f"{cfg.SEED}:{value}".encode()).hexdigest()[:8],16)/0xffffffff
    return "train" if u<.8 else ("val" if u<.9 else "test")
def yvec(labels):
    y=torch.zeros(19)
    for label in labels: y[L2I[label]]=1
    return y

raw=pd.read_csv(cfg.METADATA)
required={"patch_id","s1_name"}; missing=required-set(raw.columns)
if missing: raise ValueError(f"Metadata missing {missing}; columns={list(raw.columns)}")
label_col=next((x for x in ("labels_filtered","labels") if x in raw.columns),None)
split_col="split" if "split" in raw.columns else None
rows=[]; rejected=defaultdict(int); missing_files=[]
for index,row in tqdm(raw.iterrows(),total=len(raw),desc="Validating optical-to-optical"):
    qpath=patch_path(QUERY_MODALITY,row); gpath=patch_path(GALLERY_MODALITY,row)
    if not qpath.exists(): rejected["missing_query"]+=1; missing_files.append({"index":index,"side":"query","path":str(qpath)}); continue
    if not gpath.exists(): rejected["missing_gallery"]+=1; missing_files.append({"index":index,"side":"gallery","path":str(gpath)}); continue
    qfiles=band_map(qpath,MODALITIES[QUERY_MODALITY]["bands"]); gfiles=band_map(gpath,MODALITIES[GALLERY_MODALITY]["bands"])
    if len(qfiles)!=len(MODALITIES[QUERY_MODALITY]["bands"]): rejected["query_bands"]+=1; continue
    if len(gfiles)!=len(MODALITIES[GALLERY_MODALITY]["bands"]): rejected["gallery_bands"]+=1; continue
    labels=parse_labels(row[label_col]) if label_col else []
    if not labels: rejected["labels"]+=1; continue
    split=str(row[split_col]).lower() if split_col else split_for(row["patch_id"])
    split={"validation":"val","valid":"val","training":"train","testing":"test"}.get(split,split)
    if split not in ("train","val","test"): split=split_for(row["patch_id"])
    rows.append({"patch_id":str(row["patch_id"]),"qfiles":qfiles,"gfiles":gfiles,"labels":labels,"split":split})
meta=pd.DataFrame(rows); pd.DataFrame(missing_files).to_csv(cfg.RESULTS_DIR/"missing_files.csv",index=False)
if meta.empty: raise RuntimeError(f"No valid rows: {dict(rejected)}")
print("Input",len(raw),"valid",len(meta),"rejected",dict(rejected)); display(meta.split.value_counts())

## 3. Task-specific data pipeline

In [ ]:
def read_stack(paths,bands):
    arrays=[]
    for band in bands:
        with rasterio.open(paths[band]) as src:
            arrays.append(src.read(1,out_shape=(cfg.IMAGE_SIZE,cfg.IMAGE_SIZE),out_dtype="float32",resampling=Resampling.nearest))
    x=torch.from_numpy(np.stack(arrays)).float()
    mean=torch.tensor([MEAN[b] for b in bands])[:,None,None]; std=torch.tensor([STD[b] for b in bands])[:,None,None]
    x=(x-mean)/std
    if not torch.isfinite(x).all(): raise ValueError("Non-finite pixels")
    return x
def random_aug(x):
    if random.random()<.5: x=x.flip(-1)
    if random.random()<.5: x=x.flip(-2)
    return torch.rot90(x,random.randrange(4),(-2,-1))
def paired_aug(a,b):
    if random.random()<.5: a,b=a.flip(-1),b.flip(-1)
    if random.random()<.5: a,b=a.flip(-2),b.flip(-2)
    k=random.randrange(4); return torch.rot90(a,k,(-2,-1)),torch.rot90(b,k,(-2,-1))
class RetrievalDataset(Dataset):
    def __init__(self,frame,train=False): self.frame=frame.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]; x=read_stack(row.qfiles,MODALITIES[QUERY_MODALITY]["bands"])
        if self.train: query=random_aug(x); gallery=random_aug(x)
        else: query=x; gallery=x.clone()
        return {"query":query,"gallery":gallery,"labels":yvec(row.labels),"patch_id":row.patch_id}
def limited(frame,n): return frame.copy() if n is None or len(frame)<=n else frame.sample(n,random_state=cfg.SEED)
train_frame=limited(meta[meta.split=="train"],cfg.MAX_TRAIN_SAMPLES); val_frame=limited(meta[meta.split=="val"],cfg.MAX_EVAL_SAMPLES); test_frame=limited(meta[meta.split=="test"],cfg.MAX_EVAL_SAMPLES)
if len(val_frame)==0 or len(test_frame)==0:
    shuffled=meta.sample(frac=1,random_state=cfg.SEED).reset_index(drop=True); n=len(shuffled)
    train_frame, val_frame, test_frame=shuffled[:int(.8*n)],shuffled[int(.8*n):int(.9*n)],shuffled[int(.9*n):]
train_ds=RetrievalDataset(train_frame,True); val_ds=RetrievalDataset(val_frame); test_ds=RetrievalDataset(test_frame)
kwargs={"num_workers":cfg.NUM_WORKERS,"pin_memory":DEVICE.type=="cuda","persistent_workers":cfg.NUM_WORKERS>0}
train_loader=DataLoader(train_ds,cfg.BATCH_SIZE,shuffle=True,drop_last=len(train_ds)>=cfg.BATCH_SIZE,**kwargs)
val_loader=DataLoader(val_ds,cfg.EVAL_BATCH_SIZE,shuffle=False,**kwargs); test_loader=DataLoader(test_ds,cfg.EVAL_BATCH_SIZE,shuffle=False,**kwargs)
sample=train_ds[0]
assert sample["query"].shape==(len(MODALITIES[QUERY_MODALITY]["bands"]),120,120)
assert sample["gallery"].shape==(len(MODALITIES[GALLERY_MODALITY]["bands"]),120,120)
assert torch.isfinite(sample["query"]).all() and sample["labels"].sum()>0
print(len(train_ds),len(val_ds),len(test_ds),sample["query"].shape,sample["gallery"].shape)

## 4. 3×3 data visualization

In [ ]:
def denormalize(x,modality):
    bands=MODALITIES[modality]["bands"]; return (x*torch.tensor([STD[b] for b in bands])[:,None,None]+torch.tensor([MEAN[b] for b in bands])[:,None,None]).numpy()
def preview(x,modality):
    z=denormalize(x,modality); bands=MODALITIES[modality]["bands"]
    if modality=="sar":
        image=z[bands.index("VV")]; lo,hi=np.percentile(image,[2,98]); return np.clip((image-lo)/(hi-lo+1e-6),0,1)
    image=z[[bands.index("B04"),bands.index("B03"),bands.index("B02")]]; lo,hi=np.percentile(image,[2,98])
    return np.clip((np.moveaxis(image,0,-1)-lo)/(hi-lo+1e-6),0,1)
fig,axes=plt.subplots(3,3,figsize=(12,12))
for i,ax in enumerate(axes.flat):
    item=train_ds[i]; ax.imshow(preview(item["query"],QUERY_MODALITY),cmap="gray" if QUERY_MODALITY=="sar" else None)
    ax.set_title(f"{QUERY_MODALITY} · {item['patch_id']}"); ax.axis("off")
plt.tight_layout(); plt.savefig(cfg.RESULTS_DIR/"data_3x3.png",dpi=150,bbox_inches="tight"); plt.show()

## 5. Model building and verified pretrained weights

In [ ]:
class Encoder(nn.Module):
    def __init__(self,backbone):
        super().__init__(); self.backbone=backbone; self.projector=nn.Sequential(nn.Linear(backbone.num_features,cfg.EMBED_DIM),nn.GELU(),nn.LayerNorm(cfg.EMBED_DIM),nn.Linear(cfg.EMBED_DIM,cfg.EMBED_DIM))
    def forward(self,x): return F.normalize(self.projector(self.backbone(x)),dim=-1)

def hf_file(repo,filename,modality):
    if cfg.HF_LOCAL_DIR:
        path=Path(cfg.HF_LOCAL_DIR)/modality/filename
        if not path.exists(): raise FileNotFoundError(path)
        return str(path)
    return hf_hub_download(repo_id=repo,filename=filename,token=cfg.HF_TOKEN)
def reben_state(repo,modality):
    config=json.loads(Path(hf_file(repo,"config.json",modality)).read_text())
    if config["timm_model_name"]!=cfg.MODEL_NAME or config["image_size"]!=cfg.IMAGE_SIZE: raise ValueError(f"Incompatible reBEN config: {config}")
    raw=load_safetensors(hf_file(repo,"model.safetensors",modality),device="cpu"); state={}
    for key,value in raw.items():
        for prefix in ("model.vision_encoder.","vision_encoder."):
            if key.startswith(prefix): state[key[len(prefix):]]=value; break
    if not state: raise RuntimeError("No vision_encoder tensors found")
    return state,config
def load_reben(encoder,repo,modality,adapt_optical=False):
    state,info=reben_state(repo,modality); target=encoder.backbone.state_dict()
    if adapt_optical:
        key="patch_embed.proj.weight"; old=state[key]; legacy=["B02","B03","B04","B08","B05","B06","B07","B11","B12","B8A"]
        state[key]=old[:,[legacy.index(x) for x in ["B04","B03","B02"]]]*(10/3)
    matched={k:v for k,v in state.items() if k in target and v.shape==target[k].shape}
    coverage=sum(v.numel() for v in matched.values())/sum(v.numel() for v in target.values())
    encoder.backbone.load_state_dict(matched,strict=False)
    if coverage<.95: raise RuntimeError(f"{modality} reBEN coverage only {coverage:.1%}")
    return {"source":repo,"coverage":coverage,"matched":len(matched)}
def build_encoder(modality):
    channels=len(MODALITIES[modality]["bands"]); kwargs={"in_chans":channels,"num_classes":0,"img_size":cfg.IMAGE_SIZE}
    if modality=="optical" and cfg.OPTICAL_INIT=="imagenet":
        try: backbone=timm.create_model(cfg.MODEL_NAME,pretrained=True,**kwargs); init={"source":"ImageNet","coverage":1.0}
        except Exception as error: print("ImageNet unavailable; random fallback",error); backbone=timm.create_model(cfg.MODEL_NAME,pretrained=False,**kwargs); init={"source":"random_fallback","coverage":0.0}
    else:
        backbone=timm.create_model(cfg.MODEL_NAME,pretrained=False,**kwargs); init={"source":"random","coverage":0.0}
    encoder=Encoder(backbone)
    if modality in REBEN_REPOS: init=load_reben(encoder,REBEN_REPOS[modality],modality)
    elif modality=="optical" and cfg.OPTICAL_INIT=="adapt_reben_s2":
        init=load_reben(encoder,"BIFOLD-BigEarthNetv2-0/vit_base_patch8_224-s2-v0.1.1","s2",True)
    elif modality=="optical" and cfg.OPTICAL_INIT not in ("imagenet","random"): raise ValueError("OPTICAL_INIT must be imagenet, adapt_reben_s2, or random")
    return encoder,init
class RetrievalModel(nn.Module):
    def __init__(self): super().__init__(); self.encoder,self.initialization=build_encoder(QUERY_MODALITY)
    def forward(self,query,gallery): return self.encoder(query),self.encoder(gallery)
    def encode_query(self,x): return self.encoder(x)
    def encode_gallery(self,x): return self.encoder(x)
model=RetrievalModel().to(DEVICE); print(json.dumps(model.initialization,indent=2))

## 6. Model training

In [ ]:
def contrastive_loss(a,b,labels):
    logits=a@b.T/cfg.TEMPERATURE; targets=torch.arange(len(a),device=a.device)
    paired=(F.cross_entropy(logits,targets)+F.cross_entropy(logits.T,targets))/2
    z=torch.cat([a,b]); y=torch.cat([labels,labels]); sim=z@z.T/cfg.TEMPERATURE
    eye=torch.eye(len(z),dtype=torch.bool,device=z.device); positive=(y@y.T>0)&~eye
    sim=sim-sim.max(1,keepdim=True).values.detach(); ex=torch.exp(sim)*~eye; valid=positive.any(1)
    semantic=-torch.log(((ex*positive).sum(1).clamp_min(1e-12)/ex.sum(1).clamp_min(1e-12))[valid]).mean()
    return cfg.PAIRED_WEIGHT*paired+cfg.SEMANTIC_WEIGHT*semantic,paired,semantic
@torch.no_grad()
def validation_loss():
    model.eval(); total=count=0
    for batch in val_loader:
        q,g,y=batch["query"].to(DEVICE),batch["gallery"].to(DEVICE),batch["labels"].to(DEVICE); a,b=model(q,g); loss,_,_=contrastive_loss(a,b,y)
        total+=loss.item()*len(q); count+=len(q)
    return total/max(count,1)
def train_model():
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,cfg.EPOCHS)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda"); history=[]; start=0; best=float("inf"); stale=0
    if cfg.RESUME and LAST.exists():
        saved=torch.load(LAST,map_location=DEVICE,weights_only=False); model.load_state_dict(saved["model"]); optimizer.load_state_dict(saved["optimizer"]); scheduler.load_state_dict(saved["scheduler"]); history=saved["history"]; start=saved["epoch"]+1; best=saved["best"]
    for epoch in range(start,cfg.EPOCHS):
        model.train(); sums=defaultdict(float); count=0
        for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}"):
            q,g,y=batch["query"].to(DEVICE),batch["gallery"].to(DEVICE),batch["labels"].to(DEVICE); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=="cuda"): a,b=model(q,g); loss,paired,semantic=contrastive_loss(a,b,y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
            for k,v in {"loss":loss.item(),"paired":paired.item(),"semantic":semantic.item()}.items(): sums[k]+=v*len(q)
            count+=len(q)
        val=validation_loss(); scheduler.step(); row={"epoch":epoch+1,"train_loss":sums["loss"]/count,"paired_loss":sums["paired"]/count,"semantic_loss":sums["semantic"]/count,"val_loss":val,"lr":optimizer.param_groups[0]["lr"]}; history.append(row)
        payload={"epoch":epoch,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),"history":history,"best":min(best,val),"config":{k:str(v) if isinstance(v,Path) else v for k,v in asdict(cfg).items()},"task":TASK_NAME}
        torch.save(payload,LAST)
        if val<best: best=val; stale=0; torch.save(payload,BEST)
        else: stale+=1
        pd.DataFrame(history).to_csv(cfg.RESULTS_DIR/"history.csv",index=False); print(row)
        if stale>=cfg.PATIENCE: break
    return history
history=train_model() if cfg.RUN_TRAINING else []
if BEST.exists(): model.load_state_dict(torch.load(BEST,map_location=DEVICE,weights_only=False)["model"])
if history:
    pd.DataFrame(history).plot(x="epoch",y=["train_loss","val_loss"],marker="o",figsize=(8,4)); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(cfg.RESULTS_DIR/"training_history.png",dpi=150); plt.show()

## 7. Evaluation and metric checks

In [ ]:
@torch.no_grad()
def extract_embeddings():
    model.eval(); query=[]; gallery=[]; labels=[]; ids=[]; start=time.perf_counter()
    for batch in tqdm(test_loader,desc="Embedding"):
        query.append(model.encode_query(batch["query"].to(DEVICE)).cpu()); gallery.append(model.encode_gallery(batch["gallery"].to(DEVICE)).cpu()); labels.append(batch["labels"]); ids+=batch["patch_id"]
    return torch.cat(query),torch.cat(gallery),torch.cat(labels),ids,len(ids)/(time.perf_counter()-start)
Q,G,Y,IDS,throughput=extract_embeddings()
def evaluate():
    maxk=min(10,len(G)-(1 if SAME_MODAL else 0)); sums=defaultdict(float); predictions=[]; latency=[]; gd=G.to(DEVICE); yd=Y.to(DEVICE)
    for start in range(0,len(Q),cfg.SIM_CHUNK):
        end=min(start+cfg.SIM_CHUNK,len(Q)); q=Q[start:end].to(DEVICE); qy=Y[start:end].to(DEVICE)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        tic=time.perf_counter(); scores=q@gd.T
        if SAME_MODAL: scores[torch.arange(end-start),torch.arange(start,end,device=DEVICE)]=-torch.inf
        values,indices=scores.topk(maxk,dim=1)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        latency += [(time.perf_counter()-tic)/(end-start)]*(end-start)
        all_relevant=qy@yd.T>0
        if SAME_MODAL: all_relevant[torch.arange(end-start),torch.arange(start,end,device=DEVICE)]=False
        relevant=all_relevant.gather(1,indices); totals=all_relevant.sum(1).clamp_min(1)
        for requested in (5,10):
            k=min(requested,maxk); rel=relevant[:,:k].float(); hits=rel.sum(1); precision=hits/k; recall=hits/totals; f1=2*precision*recall/(precision+recall).clamp_min(1e-12)
            prefix=rel.cumsum(1)/torch.arange(1,k+1,device=DEVICE); ap=(prefix*rel).sum(1)/torch.minimum(totals,torch.tensor(k,device=DEVICE))
            for name,value in ((f"precision@{requested}",precision),(f"recall@{requested}",recall),(f"f1@{requested}",f1),(f"map@{requested}",ap)): sums[name]+=value.sum().item()
        for i in range(end-start): predictions.append({"query_id":IDS[start+i],"gallery_ids":[IDS[j] for j in indices[i].cpu().tolist()],"scores":values[i].cpu().tolist(),"relevant":relevant[i].cpu().tolist()})
    metrics={k:v/len(Q) for k,v in sums.items()}; metrics.update({"mean_latency_ms":1000*np.mean(latency),"median_latency_ms":1000*np.median(latency),"embedding_pairs_per_second":throughput})
    return metrics,predictions
metrics,predictions=evaluate(); display(pd.DataFrame([metrics])); pd.DataFrame([metrics]).to_csv(cfg.RESULTS_DIR/"metrics.csv",index=False)
with open(cfg.RESULTS_DIR/"predictions.jsonl","w") as f:
    for row in predictions: f.write(json.dumps(row)+"\n")
assert Q.shape==G.shape==(len(test_ds),cfg.EMBED_DIM); assert torch.isfinite(Q).all()
model.eval()
with torch.no_grad(): a=model.encode_query(sample["query"].unsqueeze(0).to(DEVICE)); b=model.encode_query(sample["query"].unsqueeze(0).to(DEVICE))
assert torch.allclose(a,b,atol=1e-6)
if SAME_MODAL:
    for row in predictions: assert row["query_id"] not in row["gallery_ids"]
if BEST.exists():
    state=torch.load(BEST,map_location="cpu",weights_only=False)["model"]; model.load_state_dict(state,strict=True)
print("Metric and checkpoint smoke tests passed.")

## 8. Qualitative retrieval and inference

In [ ]:
def load_inference_file(path,modality):
    bands=MODALITIES[modality]["bands"]; path=Path(path)
    if path.suffix.lower()==".npz":
        payload=np.load(path); array=payload["image"] if "image" in payload else payload[payload.files[0]]
    else:
        with rasterio.open(path) as src: array=src.read(out_shape=(src.count,cfg.IMAGE_SIZE,cfg.IMAGE_SIZE),out_dtype="float32",resampling=Resampling.nearest)
    array=np.asarray(array,dtype="float32")
    if array.ndim==3 and array.shape[-1]==len(bands): array=np.moveaxis(array,-1,0)
    if array.shape[0]!=len(bands): raise ValueError(f"{modality} requires ordered bands {bands}; got {array.shape}")
    if array.shape[-2:]!=(cfg.IMAGE_SIZE,cfg.IMAGE_SIZE): array=F.interpolate(torch.from_numpy(array)[None],size=(cfg.IMAGE_SIZE,cfg.IMAGE_SIZE),mode="nearest")[0].numpy()
    mean=np.array([MEAN[b] for b in bands])[:,None,None]; std=np.array([STD[b] for b in bands])[:,None,None]
    return torch.tensor((array-mean)/std,dtype=torch.float32)
@torch.no_grad()
def retrieve(query_index=0,query_file=None,top_k=10,show=True):
    model.eval()
    if query_file is None: tensor=test_ds[query_index]["query"]; query_id=IDS[query_index]
    else: tensor=load_inference_file(query_file,QUERY_MODALITY); query_id=Path(query_file).stem
    z=model.encode_query(tensor.unsqueeze(0).to(DEVICE)).cpu(); scores=(z@G.T)[0]
    if SAME_MODAL and query_file is None: scores[query_index]=-torch.inf
    k=min(top_k,len(G)-(1 if SAME_MODAL and query_file is None else 0)); values,indices=scores.topk(k)
    result=pd.DataFrame({"rank":range(1,k+1),"patch_id":[IDS[i] for i in indices],"cosine_score":values.tolist()})
    if show:
        columns=min(k+1,6); fig,axes=plt.subplots(1,columns,figsize=(3*columns,3)); axes=np.atleast_1d(axes)
        axes[0].imshow(preview(tensor,QUERY_MODALITY),cmap="gray" if QUERY_MODALITY=="sar" else None); axes[0].set_title(f"QUERY\n{query_id}"); axes[0].axis("off")
        for rank,index in enumerate(indices[:columns-1],1):
            item=test_ds[int(index)]; axes[rank].imshow(preview(item["gallery"],GALLERY_MODALITY),cmap="gray" if GALLERY_MODALITY=="sar" else None); axes[rank].set_title(f"#{rank}\n{IDS[index]}\n{values[rank-1]:.3f}"); axes[rank].axis("off")
        plt.tight_layout(); plt.savefig(cfg.RESULTS_DIR/"inference_topk.png",dpi=150,bbox_inches="tight"); plt.show()
    return result
display(retrieve(query_index=0,top_k=10))